In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
df = pd.read_parquet("..\datasets\merged.parquet")

df.head()

,smiles,ir_spectra,carboxylic_acid,amino,sulfonic_acid,guanidino,source,ir_length
0,COc1nc2ccccc2cc1C(=O)O,"[-0.004480999894440174, 0.007278999779373407, ...",1,0,0,0,mss,1800
1,C/C=C(/C(=O)O)c1nsc(NC(=O)OC(C)(C)C)n1,"[0.009829999879002571, 0.01835699938237667, 0....",1,0,0,0,mss,1800
2,CCN(CC)c1nc2cc(C(=O)O)ccc2nc1-c1ccccc1,"[0.0029329999815672636, 0.004985999781638384, ...",1,0,0,0,mss,1800
3,Nc1nnc(N2CCOCC2)c(-c2ccccc2)n1,"[0.016256000846624374, 0.006963000167161226, 0...",0,1,0,0,mss,1800
4,Cc1cc(O)c([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C...,"[0.028991999104619026, 0.08290799707174301, 0....",0,1,0,1,mss,1800


Interpolation Here

In [4]:
# Filter to only length 1800 spectra for consistent input
# temp since still no interpolation code
df = df[df['ir_length'] == 1800].reset_index(drop=True)



Take a subset bc otherwise training takes too long

In [ ]:
# Keep all sulfonic acid samples (only 68) and sample the rest
mask_rare = (df['sulfonic_acid'] == 1)
df_rare   = df[mask_rare]
df_common = df[~mask_rare].sample(n=5000, random_state=42)
df_knn    = pd.concat([df_rare, df_common]).sample(
                frac=1, random_state=42).reset_index(drop=True)

X = np.vstack(df_knn['ir_spectra'].values)
y = df_knn[['carboxylic_acid', 'amino',
            'sulfonic_acid', 'guanidino']].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=43)

In [6]:
trf = StandardScaler()
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

# Wrap KNN in MultiOutputClassifier so one KNN per functional group
knn = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=20)) #try 20 neighbors first
knn.fit(X_train_trf, y_train)
y_pred = knn.predict(X_test_trf)


print("\nPer class results:")
print(classification_report(
    y_test, y_pred,
    target_names=['carboxylic_acid', 'amino', 'sulfonic_acid', 'guanidino'],
    zero_division=0
))


Per class results:
                 precision    recall  f1-score   support

carboxylic_acid       0.95      0.94      0.94       418
          amino       0.98      0.92      0.95       612
  sulfonic_acid       0.00      0.00      0.00        15
      guanidino       1.00      0.07      0.13        14

      micro avg       0.97      0.90      0.93      1059
      macro avg       0.73      0.48      0.51      1059
   weighted avg       0.95      0.90      0.92      1059
    samples avg       0.94      0.92      0.93      1059



Trying different k values  
Could look into finding optimal k?

In [ ]:
    # Trying different k values to see if it improves results
for k in [5, 10, 20, 50]:
    knn = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=k))
    knn.fit(X_train_trf, y_train)
    y_pred = knn.predict(X_test_trf)
    
    print(f"\nk={k}")
    print(classification_report(
        y_test, y_pred,
        target_names=['carboxylic_acid', 'amino', 'sulfonic_acid', 'guanidino'],
        zero_division=0
    ))


k=5
                 precision    recall  f1-score   support

carboxylic_acid       0.94      0.92      0.93       418
          amino       0.97      0.95      0.96       612
  sulfonic_acid       0.67      0.40      0.50        15
      guanidino       1.00      0.21      0.35        14

      micro avg       0.95      0.92      0.94      1059
      macro avg       0.89      0.62      0.69      1059
   weighted avg       0.95      0.92      0.93      1059
    samples avg       0.95      0.93      0.94      1059


k=10
                 precision    recall  f1-score   support

carboxylic_acid       0.95      0.93      0.94       418
          amino       0.97      0.93      0.95       612
  sulfonic_acid       1.00      0.13      0.24        15
      guanidino       1.00      0.14      0.25        14

      micro avg       0.96      0.91      0.93      1059
      macro avg       0.98      0.53      0.59      1059
   weighted avg       0.96      0.91      0.93      1059
    samples avg